# convolution-by-integration: binned data

The convolution of binned data is somewhat different than that of event-based, as the data has a somewhat different meaning. With binned data, the value in the columns indicates the center-value of the bin (usually). As such, we need to inform the convolution about the bin edges, which internally allows us to select the matching indices for a particular bin. Moreover, because each bin spans a range of delay times, as opposed to the singular value for event-based data, these bins can fall in more than one starformation bin. We use the bin edge info to calculate, given a particular target convolution time, the overlap fraction with the star formation bins (see below).


Apart from some different configuration in the `convolution_instructions`, binned data is convolved in the same way as event-based data.

Because the binned data spans a range of delay times, we can not just use the SFR of one of the SFR bins, but rather we should calculate the average SFR by calculating how much the delay time bin overlaps with SFR bins, what the rates and bin-widths are, and then calculate a weighted average. In its most general form this is written as
$$
% Summation form of the equation
R_j^{d} = \frac{1}{\Delta \tilde{t}^{d}} \sum_{i} w_i \, R_i^{s} \, \Delta t_i^{s}
$$
Here $R_{j}^{d}$ is the averaged starformation rate over for delay-time bin $j$, $w_i$ indicates the fraction that delay-time data bin $j$ overlaps with SFR bin $i$, $R_{i}^{s}$ indicates the SFR rate in SFR bin $i$, $\Delta t_{i}^{s}$ denotes the bin width of SFR bin $i$, and $\Delta t^{d}$ indicates the cropped width of delay-time bin $j$,

$$
% Definition of cropped time step
\Delta \tilde{t}_j^{d} = \min \left( \Delta t_{j}^{d}, \sum_{i} w_i \, \Delta t_i^{s} \right).
$$

This cropped width is relevant when the delay-time bin exceeds beyond the last star formation rate bin edge.




based on the example below, can be expressed as follows

$$
% Average Star Formation Rate equation
R_1^{d} = \frac{w_0 \, R_1^{s} \, \Delta t_1^{s} + w_2 \, R_2^{s} \, \Delta t_2^{s} + w_3 \, R_3^{s} \, \Delta t_3^{s}}{\Delta \tilde{t}^{d}}
$$
More generally, this can be written as,







TODO: describe how this is done internally with fraction of bin overlap

In [27]:
import copy
import os
import h5py
import pandas as pd
import numpy as np
import json
import logging
import astropy.units as u
from syntheticstellarpopconvolve import convolve, default_convolution_config


from syntheticstellarpopconvolve.ensemble_utils import convert_ensemble_to_dataframe
from syntheticstellarpopconvolve.general_functions import calculate_bin_edges, temp_dir

#
TMP_DIR = temp_dir("notebook", "tutorial_binned_data_convolution", clean_path=True)

Lets start with setting up some dummy data. Here we use a bin-width of 1, and a bin-center of 0.25+N. 

Based on the bin-centers we can automatically figure out what the edges are with the `calculate_bin_edges` function.

In [28]:
records = [
    {"time": 0.25, "value": 10, "probability": 1},
    {"time": 1.25, "probability": 2, "value": 20},
    {"time": 2.25, "probability": 3, "value": 30},
    {"time": 0.25, "value": 11, "probability": 1.1},
    {"time": 3.25, "probability": 4, "value": 40},
]

example_dataframe = pd.DataFrame.from_records(records)

sorted_unique_time_centers = np.sort(example_dataframe["time"].unique())
time_bin_edges = calculate_bin_edges(sorted_unique_time_centers)

print('time_bin_edges', time_bin_edges)




time_bin_edges [-0.25  0.75  1.75  2.75  3.75]


We now need to set up the input file and set up the configuration, the starformation rate dictionary and the convolution steps. These steps are all unchanged with respect to non-binned data.

In [29]:
# create file
input_hdf5_filename = os.path.join(TMP_DIR, "input_hdf5.h5")
output_hdf5_filename = os.path.join(TMP_DIR, "output_hdf5.h5")
input_hdf5_file = h5py.File(input_hdf5_filename, "w")

# Create groups main
input_hdf5_file.create_group("input_data")
input_hdf5_file.create_group("config")

# close
input_hdf5_file.close()

# store the data frame in the hdf5file
example_dataframe.to_hdf(input_hdf5_filename, key="input_data/binned_example")

#
convolution_config = copy.copy(default_convolution_config)
convolution_config["input_filename"] = input_hdf5_filename
convolution_config["output_filename"] = output_hdf5_filename
convolution_config["tmp_dir"] = TMP_DIR
convolution_config["multiprocessing"] = False
convolution_config["logger"].setLevel(logging.CRITICAL)
convolution_config["time_type"] = "lookback_time"

convolution_config["convolution_lookback_time_bin_edges"] = np.arange(0, 3, 1) * u.yr

###
# set up SFR dict
sfr_dict = {}
sfr_dict["lookback_time_bin_edges"] = np.arange(0, 10, 1) * u.yr

sfr_dict["starformation_rate_array"] = (
    np.arange(0, len(sfr_dict["lookback_time_bin_edges"]) - 1) ** 2 * u.Msun / u.yr
)

# store
convolution_config["SFR_info"] = sfr_dict

#
convolution_config["multiply_by_sfr_time_binsize"] = False
convolution_config["multiply_by_convolution_time_binsize"] = False

To properly configure the code to convolve binned data, we need to indicate so in the `convolution_instructions` with the entry

```python
convolution_config["convolution_instructions"] = [
    {
    ...
    "contains_binned_data": True,
    ...
    }
]
```

and provide information about the delay time bins through 

```python
convolution_config["convolution_instructions"] = [
    {
    ...
    "delay_time_data_bin_info_dict": {
        "delay_time_data_bin_edges": time_bin_edges * u.yr
    },
    ...
    }
]
```

In [30]:
###
# convolution instructions
convolution_config["convolution_instructions"] = [
    {
        "convolution_type": "integrate",
        "input_data_name": "binned_example",
        "output_data_name": "binned_example",
        "contains_binned_data": True,
        "ignore_metallicity": True,
        "delay_time_data_bin_info_dict": {
            "delay_time_data_bin_edges": time_bin_edges * u.yr
        },
        "data_column_dict": {
            # required
            "normalized_yield": {"column_name": "probability", "unit": 1 / u.Msun},
            # "normalized_yield": "probability",
            "delay_time": {"column_name": "time", "unit": u.yr},
        },
    },
]

In [ ]:
After this, the convolution handled in the same way as non-binned/event-based data.

In [31]:
# convolve
convolve(config=convolution_config)

print("finished convolution")

# read out content and integrate until today
with h5py.File(convolution_config["output_filename"], "r") as output_hdf5_file:
    #
    main_group = "output_data/binned_example/binned_example/convolution_results/"

    ################
    #

    # loop over the formation-time bins
    formation_time_bin_keys = list(output_hdf5_file[main_group].keys())
    formation_time_bin_keys = sorted(
        formation_time_bin_keys, key=lambda x: float(x.split(" ")[0])
    )
    for formation_time_bin_key in formation_time_bin_keys:

        print("=================================")
        print(f"formation_time_bin_key: {formation_time_bin_key}")
        print("=================================")

        ###########
        # Read out data

        # convert units
        unit_dict = json.loads(
            output_hdf5_file[f"{main_group}/{formation_time_bin_key}"].attrs["units"]
        )
        unit_dict = {key: u.Unit(val) for key, val in unit_dict.items()}
        print(unit_dict)

        #
        yield_result = output_hdf5_file[f"{main_group}/{formation_time_bin_key}/yield"][
            ()
        ]

        print(yield_result)

finished convolution
formation_time_bin_key: 0.5 yr
{'yield': Unit("1 / yr")}
[ 0.25   3.5   15.75   0.275 43.   ]
formation_time_bin_key: 1.5 yr
{'yield': Unit("1 / yr")}
[ 1.75  10.5   32.25   1.925 73.   ]


## Post-processing binned data

In [ ]:
Post-processing of binned data is also handled similar to the non-binned data.